# WEEK BY WEEK WINNER PREDICTIONS

## Potential name for application: Gamelytics

In [13]:
import nfl_data_py as nfl
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from tensorflow import keras
from keras import layers

In [14]:
print(nfl.import_schedules([2026]))

              game_id  season game_type  week     gameday    weekday gametime  \
7276   2026_01_NE_SEA    2026       REG     1  2026-09-09  Wednesday    20:20   
7277    2026_01_SF_LA    2026       REG     1  2026-09-10   Thursday    20:35   
7278  2026_01_CHI_CAR    2026       REG     1  2026-09-13     Sunday    13:00   
7279   2026_01_TB_CIN    2026       REG     1  2026-09-13     Sunday    13:00   
7280   2026_01_NO_DET    2026       REG     1  2026-09-13     Sunday    13:00   
...               ...     ...       ...   ...         ...        ...      ...   
7543  2026_18_CHI_MIN    2026       REG    18  2027-01-10     Sunday    13:00   
7544   2026_18_MIA_NE    2026       REG    18  2027-01-10     Sunday    13:00   
7545    2026_18_TB_NO    2026       REG    18  2027-01-10     Sunday    13:00   
7546  2026_18_PHI_NYG    2026       REG    18  2027-01-10     Sunday    13:00   
7547  2026_18_DAL_WAS    2026       REG    18  2027-01-10     Sunday    13:00   

     away_team  away_score 

## 1. Load the schedule data

In [15]:
train_seasons = list(range(2020, 2026))

predict_season = 2026

sched = nfl.import_schedules(train_seasons)

sched_pred = nfl.import_schedules([predict_season])

## 2. Compute team/season features

In [16]:
team_stats = []

for season in train_seasons:
    # get games from this season
    season_games = sched[sched['season'] == season]

    # get all teams that played this season
    all_teams = list(season_games['home_team'].unique()) + list(season_games['away_team'].unique())
    all_teams = list(set(all_teams))  # remove duplicates

    for team in all_teams:
        # games where this team was home or away
        home_games = season_games[season_games['home_team'] == team]
        away_games = season_games[season_games['away_team'] == team]

        # total points scored and allowed
        points_for = home_games['home_score'].sum() + away_games['away_score'].sum()
        points_against = home_games['away_score'].sum() + away_games['home_score'].sum()

        # count wins and losses
        home_wins = (home_games['home_score'] > home_games['away_score']).sum()
        away_wins = (away_games['away_score'] > away_games['home_score']).sum()
        wins = int(home_wins + away_wins)

        home_losses = (home_games['home_score'] < home_games['away_score']).sum()
        away_losses = (away_games['away_score'] < away_games['home_score']).sum()
        losses = int(home_losses + away_losses)

        games_played = len(home_games) + len(away_games)

        # save results
        team_stats.append({
            'season': season,
            'team': team,
            'points_for': points_for,
            'points_against': points_against,
            'wins': wins,
            'losses': losses,
            'games_played': games_played
        })

# make into dataframe
team_feats = pd.DataFrame(team_stats)

# averages
team_feats['avg_points_for'] = team_feats['points_for'] / team_feats['games_played'].replace(0, 1)
team_feats['avg_points_against'] = team_feats['points_against'] / team_feats['games_played'].replace(0, 1)

## 3. Build the matchup dataset for training

In [17]:
matchups = []
for _, g in sched.iterrows():
    try:
        home = g['home_team']
        away = g['away_team']
        season = g['season']
        week = g['week'] if 'week' in g else np.nan

        hf = team_feats[(team_feats['team']==home) & (team_feats['season']==season)].iloc[0]
        af = team_feats[(team_feats['team']==away) & (team_feats['season']==season)].iloc[0]

        matchups.append({
            'season': season,
            'week': week,
            'matchup': f"{away} @ {home}",
            'home_team': home,
            'away_team': away,
            'points_for_diff': hf['avg_points_for'] - af['avg_points_for'],
            'points_against_diff': hf['avg_points_against'] - af['avg_points_against'],
            'wins_diff': hf['wins'] - af['wins'],
            'label': 1 if g['home_score'] > g['away_score'] else 0
        })
    except:
        continue

matchups_df = pd.DataFrame(matchups)

feature_cols = ['points_for_diff','points_against_diff','wins_diff']
X = matchups_df[feature_cols].values
y = matchups_df['label'].values

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

## 4. Train Models

In [18]:
lr = LogisticRegression(max_iter=2000)
lr.fit(X_tr, y_tr)

# Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_tr, y_tr)

DecisionTreeClassifier(random_state=42)

In [19]:
# Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_tr, y_tr)

RandomForestClassifier(n_estimators=200, random_state=42)

In [20]:
# XGBoost
xgbc = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgbc.fit(X_tr, y_tr)

/opt/anaconda3/envs/tensorflowEnv/lib/python3.8/site-packages/xgboost/core.py:158: UserWarning: [16:17:33] WARNING: /var/folders/k1/30mswbxs7r1g6zwn8y4fyt500000gp/T/abs_d9k8pmaj4_/croot/xgboost-split_1724073758172/work/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, random_state=42, ...)

In [21]:
# Neural Network
def make_nn(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

nn = make_nn(X_tr.shape[1])
nn.fit(X_tr, y_tr, epochs=25, batch_size=64, validation_data=(X_val, y_val), verbose=0)

## 5. Ensemble Prediction

In [22]:
models = {'Logistic': lr, 'DecisionTree': dt, 'RandomForest': rf, 'XGBoost': xgbc, 'NeuralNet': nn}
val_scores = {k: (v.score(X_val,y_val) if k != 'NeuralNet' else v.evaluate(X_val,y_val,verbose=0)[1]) for k,v in models.items()}

def ensemble_predict(X_input):
    preds = []
    for name, mdl in models.items():
        if name == 'NeuralNet':
            p = mdl.predict(X_input).flatten()
        else:
            p = mdl.predict_proba(X_input)[:,1]
        preds.append(p)
    preds = np.vstack(preds)
    accs = np.array([val_scores[nm] for nm in models.keys()])
    weights = accs / accs.sum()
    weighted = np.dot(weights, preds)
    return weighted

## 6. Predict 2025 matchups

In [23]:
current_week = 1  # change this to the actual current week

# Build features for prediction season using 2024 team feats
feat_2025 = team_feats[team_feats['season']==2025].copy()
pred_rows = []
for _, g in sched_pred.iterrows():
    try:
        if g['week'] != current_week:
            continue  # skip other weeks
        home = g['home_team']
        away = g['away_team']
        hf = feat_2025[feat_2025['team']==home].iloc[0]
        af = feat_2025[feat_2025['team']==away].iloc[0]

        pred_rows.append({
            'matchup': f"{away} @ {home}",
            'home_team': home,
            'away_team': away,
            'points_for_diff': hf['avg_points_for'] - af['avg_points_for'],
            'points_against_diff': hf['avg_points_against'] - af['avg_points_against'],
            'wins_diff': hf['wins'] - af['wins']
        })
    except:
        continue

pred_df = pd.DataFrame(pred_rows)
X_pred = pred_df[feature_cols].values
weighted_probs = ensemble_predict(X_pred)

# Predicted winner
pred_df['predicted_winner'] = np.where(weighted_probs >= 0.5, pred_df['home_team'], pred_df['away_team'])

# Confidence as percentage
pred_df['confidence'] = np.where(
    pred_df['predicted_winner'] == pred_df['home_team'],
    weighted_probs * 100,
    (1 - weighted_probs) * 100
)

# Round for cleaner display
pred_df['confidence'] = pred_df['confidence'].round(1)

1/1 [==============================] - 0s 27ms/step


## 7. Output current weeks predictions

In [24]:
current_week_out = pred_df[['matchup','predicted_winner','confidence']]
current_week_out


,matchup,predicted_winner,confidence
0,NE @ SEA,NE,66.8
1,SF @ LA,SF,60.9
2,CHI @ CAR,CHI,64.7
3,TB @ CIN,TB,70.8
4,NO @ DET,DET,82.5
5,BUF @ HOU,HOU,66.9
6,BAL @ IND,BAL,57.7
7,CLE @ JAX,JAX,95.6
8,ATL @ PIT,PIT,72.8
9,NYJ @ TEN,TEN,61.3
